In [17]:
!pip install pyreadstat tqdm google-api-python-client google-auth google-auth-oauthlib google-auth-httplib2


In [18]:
import os
import json
import hashlib
import requests
import zipfile
import time
import pandas as pd
from tqdm import tqdm
from datetime import datetime

from google.colab import auth
from googleapiclient.discovery import build


In [19]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
auth.authenticate_user()
drive_service = build("drive", "v3")

SCRIPT_VERSION = "stage1-ingestion-v2.0"
OFFICIAL_MICRODATA_PORTAL = "https://proyectos.inei.gob.pe/microdatos/"
OFFICIAL_SELECTED_PACKAGE_TYPE = "SPSS ZIP"
SELECTED_DOWNLOAD_FORMAT = "SPSS"
ALTERNATIVE_FORMATS_AVAILABLE = ["CSV ZIP", "Stata ZIP"]
ALTERNATIVE_FORMATS_NOT_SELECTED_REASON = (
    "SPSS ZIP was selected because .sav preserves variable labels, value labels, "
    "and coding metadata required for reproducible interpretation. CSV/Stata were "
    "not selected as primary source formats."
)

_drive_cache = {}

def _drive_escape(value):
    return value.replace("\\", "\\\\").replace("'", "\\'")

def find_drive_child(parent_id, name):
    cache_key = (parent_id, name)
    if cache_key in _drive_cache:
        return _drive_cache[cache_key]

    query = f"name = '{_drive_escape(name)}' and '{parent_id}' in parents and trashed = false"
    response = drive_service.files().list(
        q=query,
        fields="files(id,name,mimeType,size,webViewLink)",
        pageSize=10,
        supportsAllDrives=True,
        includeItemsFromAllDrives=True,
    ).execute()

    files = response.get("files", [])
    item = files[0] if files else None
    _drive_cache[cache_key] = item
    return item

def drive_item_for_path(path):
    """Return Drive metadata for a path under /content/drive/MyDrive, if found."""
    rel_path = os.path.relpath(path, "/content/drive/MyDrive")
    if rel_path.startswith(".."):
        return None

    parts = [p for p in rel_path.replace("\\", "/").split("/") if p]
    parent_id = "root"
    item = None

    for part in parts:
        item = find_drive_child(parent_id, part)
        if item is None:
            return None
        parent_id = item["id"]

    return item

def drive_id_for_path(path):
    item = drive_item_for_path(path)
    return item.get("id") if item else None


In [20]:
ROOT = "/content/drive/MyDrive/ENARES_2024_PROJECT"

RAW_DIR = f"{ROOT}/01BasesDatosPrimarias"
LOG_DIR = f"{ROOT}/05Resultados/logs"

In [21]:
base_url = "https://proyectos.inei.gob.pe/iinei/srienaho/descarga/SPSS/976-Modulo{}.zip"

modules = []

for i in range(1941, 1963):  # 1962 inclusive
    modules.append({
        "module": f"Modulo{i}",
        "url": base_url.format(i)
    })



In [22]:
def sha256_file(path):
    sha256 = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            sha256.update(chunk)

    return sha256.hexdigest()

In [23]:
def download_file(url, output_path, retries=3):
    if os.path.exists(output_path):
        print(f"Already exists: {output_path}")
        return

    for attempt in range(retries):
        try:
            r = requests.get(url, stream=True, timeout=30)
            r.raise_for_status()

            with open(output_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)

            print(f"Downloaded: {output_path}")
            return

        except Exception as e:
            print(f"Attempt {attempt+1} failed: {e}")
            time.sleep(2 ** attempt)

    raise Exception(f"Failed to download: {url}")

In [24]:
def extract_zip(zip_path, extract_to):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)

In [25]:
manifest = []
catalog = []
log_lines = []

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

In [26]:
for m in modules:

    module_name = m["module"]
    module_number = module_name.replace("Modulo", "")
    module_id = f"976-Modulo{module_number}"
    url = m["url"]

    module_folder = os.path.join(RAW_DIR, module_name)
    os.makedirs(module_folder, exist_ok=True)
    drive_folder_id = drive_id_for_path(module_folder)

    zip_path = os.path.join(module_folder, f"{module_name}.zip")

    log_lines.append(f"Processing {module_name}")

    # Download or reuse cached official ZIP.
    try:
        download_file(url, zip_path)
    except Exception as e:
        log_lines.append(f"FAILED download {module_name}: {str(e)}")
        manifest.append({
            "official_microdata_portal": OFFICIAL_MICRODATA_PORTAL,
            "module_id": module_id,
            "selected_download_format": SELECTED_DOWNLOAD_FORMAT,
            "official_selected_package_type": OFFICIAL_SELECTED_PACKAGE_TYPE,
            "official_zip_file": os.path.basename(zip_path),
            "source_url": url,
            "zip_drive_id": drive_id_for_path(zip_path),
            "zip_sha256": None,
            "zip_size_bytes": None,
            "status": "failed_download",
            "drive_folder_id": drive_folder_id,
            "extracted_data_file": None,
            "extracted_data_format": None,
            "questionnaire_pdf": None,
            "variable_dictionary_pdf": None,
            "extracted_file_drive_id": None,
            "extracted_file_sha256": None,
            "extracted_file_size_bytes": None,
            "extracted_files": [],
            "alternative_formats_available": ALTERNATIVE_FORMATS_AVAILABLE,
            "alternative_formats_not_selected_reason": ALTERNATIVE_FORMATS_NOT_SELECTED_REASON,
            "timestamp": timestamp,
            "script_version": SCRIPT_VERSION,
            "error": str(e),
        })
        continue

    zip_hash = sha256_file(zip_path)
    zip_size = os.path.getsize(zip_path)
    zip_drive_id = drive_id_for_path(zip_path)

    # Extract official ZIP while keeping the ZIP intact.
    extract_folder = os.path.join(module_folder, "extracted")
    os.makedirs(extract_folder, exist_ok=True)
    extract_zip(zip_path, extract_folder)

    extracted_records = []

    for root, _, files in os.walk(extract_folder):
        for file in sorted(files):
            full_path = os.path.join(root, file)
            file_hash = sha256_file(full_path)
            file_size = os.path.getsize(full_path)
            ext = file.split(".")[-1].lower()
            file_drive_id = drive_id_for_path(full_path)

            role = "raw_extracted"
            lower_file = file.lower()
            if ext == "sav":
                role = "primary_raw_sav"
            elif "diccionario" in lower_file and ext == "pdf":
                role = "variable_dictionary_pdf"
            elif ext == "pdf":
                role = "questionnaire_pdf"

            record = {
                "file_name": file,
                "relative_path": os.path.relpath(full_path, module_folder).replace(os.sep, "/"),
                "extension": ext,
                "file_role": role,
                "drive_id": file_drive_id,
                "sha256": file_hash,
                "size_bytes": file_size,
            }
            extracted_records.append(record)

            catalog.append({
                "modulo": module_id,
                "archivo": file,
                "extension": ext,
                "file_role": role,
                "MB": file_size / (1024 * 1024),
                "drive_id": file_drive_id,
                "sha256": file_hash,
            })

    sav_files = [r for r in extracted_records if r["extension"] == "sav"]
    questionnaire_pdfs = [r for r in extracted_records if r["file_role"] == "questionnaire_pdf"]
    dictionary_pdfs = [r for r in extracted_records if r["file_role"] == "variable_dictionary_pdf"]
    primary_sav = sav_files[0] if sav_files else None

    manifest.append({
        "official_microdata_portal": OFFICIAL_MICRODATA_PORTAL,
        "module_id": module_id,
        "selected_download_format": SELECTED_DOWNLOAD_FORMAT,
        "official_selected_package_type": OFFICIAL_SELECTED_PACKAGE_TYPE,
        "official_zip_file": os.path.basename(zip_path),
        "source_url": url,
        "zip_drive_id": zip_drive_id,
        "zip_sha256": zip_hash,
        "zip_size_bytes": zip_size,
        "status": "success",
        "drive_folder_id": drive_folder_id,
        "extracted_data_file": primary_sav["file_name"] if primary_sav else None,
        "extracted_data_format": "sav" if primary_sav else None,
        "questionnaire_pdf": questionnaire_pdfs[0]["file_name"] if questionnaire_pdfs else None,
        "variable_dictionary_pdf": dictionary_pdfs[0]["file_name"] if dictionary_pdfs else None,
        "extracted_file_drive_id": primary_sav["drive_id"] if primary_sav else None,
        "extracted_file_sha256": primary_sav["sha256"] if primary_sav else None,
        "extracted_file_size_bytes": primary_sav["size_bytes"] if primary_sav else None,
        "extracted_files": extracted_records,
        "alternative_formats_available": ALTERNATIVE_FORMATS_AVAILABLE,
        "alternative_formats_not_selected_reason": ALTERNATIVE_FORMATS_NOT_SELECTED_REASON,
        "timestamp": timestamp,
        "script_version": SCRIPT_VERSION,
    })

    log_lines.append(f"SUCCESS {module_name}")


Already exists: /content/drive/MyDrive/ENARES_2024_PROJECT/01BasesDatosPrimarias/Modulo1941/Modulo1941.zip
Already exists: /content/drive/MyDrive/ENARES_2024_PROJECT/01BasesDatosPrimarias/Modulo1942/Modulo1942.zip
Already exists: /content/drive/MyDrive/ENARES_2024_PROJECT/01BasesDatosPrimarias/Modulo1943/Modulo1943.zip
Already exists: /content/drive/MyDrive/ENARES_2024_PROJECT/01BasesDatosPrimarias/Modulo1944/Modulo1944.zip
Already exists: /content/drive/MyDrive/ENARES_2024_PROJECT/01BasesDatosPrimarias/Modulo1945/Modulo1945.zip
Already exists: /content/drive/MyDrive/ENARES_2024_PROJECT/01BasesDatosPrimarias/Modulo1946/Modulo1946.zip
Already exists: /content/drive/MyDrive/ENARES_2024_PROJECT/01BasesDatosPrimarias/Modulo1947/Modulo1947.zip
Already exists: /content/drive/MyDrive/ENARES_2024_PROJECT/01BasesDatosPrimarias/Modulo1948/Modulo1948.zip
Already exists: /content/drive/MyDrive/ENARES_2024_PROJECT/01BasesDatosPrimarias/Modulo1949/Modulo1949.zip
Already exists: /content/drive/MyDriv

In [27]:
manifest_path = os.path.join(RAW_DIR, f"ENARES_2024_STAGE1_manifest_{timestamp}.json")

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=4)

print("Manifest saved:", manifest_path)

Manifest saved: /content/drive/MyDrive/ENARES_2024_PROJECT/01BasesDatosPrimarias/ENARES_2024_STAGE1_manifest_20260515_160941.json


In [ ]:
log_path = os.path.join(RAW_DIR, f"ENARES_2024_STAGE1_log_ingesta_{timestamp}.txt")

with open(log_path, "w") as f:
    f.write("\n".join(log_lines))

print("Log saved:", log_path)

Log saved: /content/drive/MyDrive/ENARES_2024_PROJECT/05Resultados/logs/ENARES_2024_STAGE1_log_ingesta_20260515_160941.txt


In [29]:
catalog_df = pd.DataFrame(catalog)
expected_catalog_columns = ["modulo", "archivo", "extension", "file_role", "MB", "drive_id", "sha256"]
catalog_df = catalog_df.reindex(columns=expected_catalog_columns)

catalog_path = os.path.join(LOG_DIR, f"ENARES_2024_STAGE1_catalogo_modulos.csv")

catalog_df.to_csv(catalog_path, index=False)

print("Catalogue saved:", catalog_path)


Catalogue saved: /content/drive/MyDrive/ENARES_2024_PROJECT/05Resultados/logs/ENARES_2024_STAGE1_catalogo_modulos.csv
